## Построение master dataset для EDA и прогнозирвоания

В данной тетради выполняется загрузка, очистка и объединение
энергетических данных из PostgreSQL в единый master dataset
для последующего анализа и построения моделей прогнозирования
цен на электроэнергию в Нидерландах.

### Основная цель:
сформировать согласованный почасовой временной ряд (hourly time series) за период 2021–2025, содержащий:

• day-ahead prices (NL/BE/DE/FR)

• imbalance prices

• load и load forecasts 

• solar/wind generation и forecasts

• cross-border flows

• weather actuals и weather forecasts

• commodity prices (gas)

### В процессе:

• данные загружаются напрямую из PostgreSQL

• timestamps приводятся к UTC

• удаляются дубликаты

• выполняется ресемплинг 15-min в hourly

• проверяется временная консистентность

• анализируются пропуски

• все таблицы объединяются в единый master dataframe

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import warnings
import pandas as pd
from src.db.connection import get_engine

warnings.filterwarnings('ignore')

In [152]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(project_root)

/Users/elenaibraeva/Desktop/electricity-price-forecasting-nl


Подключимся к базе данных и посмотрим какие таблицы есть после загрузки данных:

In [154]:
engine = get_engine()

tables = pd.read_sql(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
    """,
    engine,
)

print(tables)

                              table_name
0          raw_be_day_ahead_price_hourly
1                    raw_co2_price_15min
2           raw_crossborder_flows_hourly
3          raw_de_day_ahead_price_hourly
4          raw_fr_day_ahead_price_hourly
5                    raw_gas_price_15min
6              raw_imbalance_price_15min
7           raw_installed_capacity_15min
8                         raw_load_15min
9                raw_load_forecast_15min
10         raw_nl_day_ahead_price_hourly
11              raw_solar_forecast_15min
12            raw_solar_generation_15min
13              raw_tennet_balance_delta
14            raw_tennet_frr_activations
15         raw_tennet_metered_injections
16  raw_tennet_settled_imbalance_volumes
17          raw_tennet_settlement_prices
18             raw_weather_actual_hourly
19           raw_weather_forecast_hourly
20               raw_wind_forecast_15min
21             raw_wind_generation_15min


Сначала загружаем из базы данных таблицу с почасовыми значениями цены nl_day_ahead_price, которая будет использоваться как таргет.
Затем фильтруем данные за период с 1 января 2023 года по 31 декабря 2025 года.
После этого преобразуем столбец timestamp в формат datetime и устанавливаем его в качестве индекса, чтобы получить корректный временной ряд (time series), отсортированный по времени.

In [138]:
import pandas as pd
from src.db.connection import get_engine

engine = get_engine()

tables = {
    "be": ("raw_be_day_ahead_price_hourly", "be_day_ahead_price"),
    "de": ("raw_de_day_ahead_price_hourly", "de_day_ahead_price"),
    "fr": ("raw_fr_day_ahead_price_hourly", "fr_day_ahead_price"),
    "nl": ("raw_nl_day_ahead_price_hourly", "nl_day_ahead_price"),
}

dfs = {}

# загрузка
for country, (table, col) in tables.items():
    print(f"Loading {table}...")

    df = pd.read_sql(
        f"""
        SELECT timestamp, {col}
        FROM {table}
        WHERE timestamp >= '2021-01-01'
          AND timestamp < '2026-01-01'
        ORDER BY timestamp
        """,
        engine,
    )

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

    df = (
        df
        .drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    dfs[country] = df

# merge (начинаем с NL как базовой)
master = dfs["nl"]

for country in ["be", "de", "fr"]:
    master = master.merge(
        dfs[country],
        on="timestamp",
        how="outer"
    )

master = master.sort_values("timestamp").reset_index(drop=True)

master.head()

Loading raw_be_day_ahead_price_hourly...
Loading raw_de_day_ahead_price_hourly...
Loading raw_fr_day_ahead_price_hourly...
Loading raw_nl_day_ahead_price_hourly...


,timestamp,nl_day_ahead_price,be_day_ahead_price,de_day_ahead_price,fr_day_ahead_price
0,2021-01-01 00:00:00+00:00,48.19,48.19,48.19,48.19
1,2021-01-01 01:00:00+00:00,44.68,44.68,44.68,44.68
2,2021-01-01 02:00:00+00:00,42.92,42.92,42.92,42.92
3,2021-01-01 03:00:00+00:00,40.39,40.39,40.39,40.39
4,2021-01-01 04:00:00+00:00,40.20,40.20,40.20,40.20


In [139]:
import pandas as pd
from src.db.connection import get_engine

engine = get_engine()

tables = {
    "co2": ("raw_co2_price_15min", "co2_price"),
    "gas": ("raw_gas_price_15min", "gas_price"),
}

dfs = {}

for name, (table, col) in tables.items():
    print(f"Loading {table}...")

    df = pd.read_sql(
        f"""
        SELECT *
        FROM {table}
        WHERE timestamp >= '2021-01-01'
          AND timestamp < '2026-01-01'
        ORDER BY timestamp
        """,
        engine,
    )

    print("Columns:", df.columns.tolist())

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

    if col not in df.columns:
        raise ValueError(f"{col} not found in {table}")

    df = (
        df[["timestamp", col]]
        .drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    dfs[name] = df

#co2_df = dfs["co2"]
gas_df = dfs["gas"]

# CO2 -> hourly
#co2_hourly = (
#    co2_df
#    .set_index("timestamp")
#    .resample("1h")
#    .mean()
#    .reset_index()
#)

# GAS -> hourly
gas_hourly = (
    gas_df
    .set_index("timestamp")
    .resample("1h")
    .mean()

    .reset_index()
)

#co2_hourly.head(), 
gas_hourly.head()

Loading raw_co2_price_15min...
Columns: ['timestamp', 'co2_price', 'source', 'ticker', 'created_at']
Loading raw_gas_price_15min...
Columns: ['timestamp', 'gas_price', 'gas_price_log', 'source', 'ticker', 'created_at']


,timestamp,gas_price
0,2021-01-01 00:00:00+00:00,19.125
1,2021-01-01 01:00:00+00:00,19.125
2,2021-01-01 02:00:00+00:00,19.125
3,2021-01-01 03:00:00+00:00,19.125
4,2021-01-01 04:00:00+00:00,19.125


In [140]:
#master = master.merge(
#    co2_hourly,
#    on="timestamp",
#    how="left",
#)

master = master.merge(
    gas_hourly,
    on="timestamp",
    how="left",
)

master = master.sort_values("timestamp").reset_index(drop=True)

master.head()

,timestamp,nl_day_ahead_price,be_day_ahead_price,de_day_ahead_price,fr_day_ahead_price,gas_price
0,2021-01-01 00:00:00+00:00,48.19,48.19,48.19,48.19,19.125
1,2021-01-01 01:00:00+00:00,44.68,44.68,44.68,44.68,19.125
2,2021-01-01 02:00:00+00:00,42.92,42.92,42.92,42.92,19.125
3,2021-01-01 03:00:00+00:00,40.39,40.39,40.39,40.39,19.125
4,2021-01-01 04:00:00+00:00,40.20,40.20,40.20,40.20,19.125


In [141]:
# 1. Проверка: все ли timestamps кратны часу
not_hourly = master[
    (master["timestamp"].dt.minute != 0) |
    (master["timestamp"].dt.second != 0)
]

print("Non-hourly rows:", len(not_hourly))
not_hourly.head()

Non-hourly rows: 0


,timestamp,nl_day_ahead_price,be_day_ahead_price,de_day_ahead_price,fr_day_ahead_price,gas_price


In [142]:
flows = pd.read_sql(
    """
    SELECT
        timestamp,
        flow_nl_to_de,
        flow_de_to_nl,
        net_flow_de_nl,
        flow_nl_to_be,
        flow_be_to_nl,
        net_flow_be_nl
    FROM raw_crossborder_flows_hourly
    WHERE timestamp >= '2021-01-01'
      AND timestamp < '2026-01-01'
    ORDER BY timestamp
    """,
    engine,
)

flows["timestamp"] = pd.to_datetime(flows["timestamp"], utc=True)

flow_cols = [
    "flow_nl_to_de",
    "flow_de_to_nl",
    "net_flow_de_nl",
    "flow_nl_to_be",
    "flow_be_to_nl",
    "net_flow_be_nl",
]

for col in flow_cols:
    flows[col] = pd.to_numeric(flows[col], errors="coerce")

print("Before aggregation:")
print(flows["timestamp"].diff().value_counts().head(10))

flows_hourly = (
    flows
    .set_index("timestamp")
    .sort_index()
    .resample("1h")
    .mean()
    .dropna(how="all")
    .reset_index()
)

print("After aggregation:")
print(flows_hourly["timestamp"].diff().value_counts().head(10))

flows_hourly.head()

Before aggregation:
timestamp
0 days 00:15:00    154843
0 days 01:00:00      5110
0 days 00:30:00         4
Name: count, dtype: int64
After aggregation:
timestamp
0 days 01:00:00    43822
Name: count, dtype: int64


,timestamp,flow_nl_to_de,flow_de_to_nl,net_flow_de_nl,flow_nl_to_be,flow_be_to_nl,net_flow_be_nl
0,2021-01-01 00:00:00+00:00,0.0,2193.3700,2193.3700,1418.4175,0.0,-1418.4175
1,2021-01-01 01:00:00+00:00,0.0,2711.7125,2711.7125,1371.6950,0.0,-1371.6950
2,2021-01-01 02:00:00+00:00,0.0,2661.7350,2661.7350,662.2700,0.0,-662.2700
3,2021-01-01 03:00:00+00:00,0.0,2469.2525,2469.2525,334.5325,0.0,-334.5325
4,2021-01-01 04:00:00+00:00,0.0,2749.8375,2749.8375,626.7025,0.0,-626.7025


In [143]:
flows_raw = pd.read_sql(
    """
    SELECT
        timestamp,
        flow_nl_to_de,
        flow_de_to_nl,
        net_flow_de_nl,
        flow_nl_to_be,
        flow_be_to_nl,
        net_flow_be_nl
    FROM raw_crossborder_flows_hourly
    WHERE timestamp >= '2021-01-01'
      AND timestamp < '2026-01-01'
    ORDER BY timestamp
    """,
    engine,
)

flows_raw["timestamp"] = pd.to_datetime(flows_raw["timestamp"], utc=True)

flow_cols = [
    "flow_nl_to_de",
    "flow_de_to_nl",
    "net_flow_de_nl",
    "flow_nl_to_be",
    "flow_be_to_nl",
    "net_flow_be_nl",
]

for col in flow_cols:
    flows_raw[col] = pd.to_numeric(flows_raw[col], errors="coerce")

# Check source frequency
print(flows_raw["timestamp"].diff().value_counts().head(10))

# Aggregate to hourly
flows_hourly = (
    flows_raw
    .set_index("timestamp")
    .sort_index()
    .resample("1h")
    .mean()
    .reset_index()
)

flows_hourly.head()

timestamp
0 days 00:15:00    154843
0 days 01:00:00      5110
0 days 00:30:00         4
Name: count, dtype: int64


,timestamp,flow_nl_to_de,flow_de_to_nl,net_flow_de_nl,flow_nl_to_be,flow_be_to_nl,net_flow_be_nl
0,2021-01-01 00:00:00+00:00,0.0,2193.3700,2193.3700,1418.4175,0.0,-1418.4175
1,2021-01-01 01:00:00+00:00,0.0,2711.7125,2711.7125,1371.6950,0.0,-1371.6950
2,2021-01-01 02:00:00+00:00,0.0,2661.7350,2661.7350,662.2700,0.0,-662.2700
3,2021-01-01 03:00:00+00:00,0.0,2469.2525,2469.2525,334.5325,0.0,-334.5325
4,2021-01-01 04:00:00+00:00,0.0,2749.8375,2749.8375,626.7025,0.0,-626.7025


In [144]:
master = master.merge(
    flows_hourly,
    on="timestamp",
    how="left",
)

master = master.sort_values("timestamp").reset_index(drop=True)

master.head()

,timestamp,nl_day_ahead_price,be_day_ahead_price,de_day_ahead_price,fr_day_ahead_price,gas_price,flow_nl_to_de,flow_de_to_nl,net_flow_de_nl,flow_nl_to_be,flow_be_to_nl,net_flow_be_nl
0,2021-01-01 00:00:00+00:00,48.19,48.19,48.19,48.19,19.125,0.0,2193.3700,2193.3700,1418.4175,0.0,-1418.4175
1,2021-01-01 01:00:00+00:00,44.68,44.68,44.68,44.68,19.125,0.0,2711.7125,2711.7125,1371.6950,0.0,-1371.6950
2,2021-01-01 02:00:00+00:00,42.92,42.92,42.92,42.92,19.125,0.0,2661.7350,2661.7350,662.2700,0.0,-662.2700
3,2021-01-01 03:00:00+00:00,40.39,40.39,40.39,40.39,19.125,0.0,2469.2525,2469.2525,334.5325,0.0,-334.5325
4,2021-01-01 04:00:00+00:00,40.20,40.20,40.20,40.20,19.125,0.0,2749.8375,2749.8375,626.7025,0.0,-626.7025


In [145]:
weather_actual = pd.read_sql(
    """
    SELECT
        timestamp,
        temperature_c,
        wind_ms,
        solar_radiation,
        cloud_cover,
        precipitation,
        humidity
    FROM raw_weather_actual_hourly
    WHERE timestamp >= '2021-01-01'
      AND timestamp < '2026-01-01'
    ORDER BY timestamp
    """,
    engine,
)

weather_actual["timestamp"] = pd.to_datetime(weather_actual["timestamp"], utc=True)

weather_cols = [
    "temperature_c",
    "wind_ms",
    "solar_radiation",
    "cloud_cover",
    "precipitation",
    "humidity",
]

for col in weather_cols:
    weather_actual[col] = pd.to_numeric(weather_actual[col], errors="coerce")

# просто merge
master = master.merge(
    weather_actual,
    on="timestamp",
    how="left",
)

master = master.sort_values("timestamp").reset_index(drop=True)

In [146]:
weather_forecast = pd.read_sql(
    """
    SELECT
        timestamp,
        target_temp_forecast,
        target_wind_speed_forecast,
        target_cloud_cover_forecast,
        target_precip_forecast,
        target_shortwave_radiation_forecast,
        target_surface_pressure_forecast,
        target_wind_gusts_forecast,
        target_dew_point_forecast,
        target_relative_humidity_forecast
    FROM raw_weather_forecast_hourly
    WHERE timestamp >= '2019-01-01'
      AND timestamp < '2026-01-01'
    ORDER BY timestamp
    """,
    engine,
)

weather_forecast["timestamp"] = pd.to_datetime(weather_forecast["timestamp"], utc=True)

weather_forecast = weather_forecast.rename(columns={
    "target_temp_forecast": "temperature_forecast",
    "target_wind_speed_forecast": "wind_speed_forecast",
    "target_cloud_cover_forecast": "cloud_cover_forecast",
    "target_precip_forecast": "precipitation_forecast",
    "target_shortwave_radiation_forecast": "solar_radiation_forecast",
    "target_surface_pressure_forecast": "pressure_forecast",
    "target_wind_gusts_forecast": "wind_gusts_forecast",
    "target_dew_point_forecast": "dew_point_forecast",
    "target_relative_humidity_forecast": "humidity_forecast",
})

forecast_cols = [
    "temperature_forecast",
    "wind_speed_forecast",
    "cloud_cover_forecast",
    "precipitation_forecast",
    "solar_radiation_forecast",
    "pressure_forecast",
    "wind_gusts_forecast",
    "dew_point_forecast",
    "humidity_forecast",
]

for col in forecast_cols:
    weather_forecast[col] = pd.to_numeric(weather_forecast[col], errors="coerce")

weather_forecast = (
    weather_forecast
    .drop_duplicates(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

master = master.drop(
    columns=[c for c in forecast_cols if c in master.columns],
    errors="ignore",
)

master = master.merge(
    weather_forecast,
    on="timestamp",
    how="left",
)

master = master.sort_values("timestamp").reset_index(drop=True)

master[forecast_cols].isna().mean()

temperature_forecast        0.0
wind_speed_forecast         0.0
cloud_cover_forecast        0.0
precipitation_forecast      0.0
solar_radiation_forecast    0.0
pressure_forecast           0.0
wind_gusts_forecast         0.0
dew_point_forecast          0.0
humidity_forecast           0.0
dtype: float64

In [147]:
master.head()

,timestamp,nl_day_ahead_price,be_day_ahead_price,de_day_ahead_price,fr_day_ahead_price,gas_price,flow_nl_to_de,flow_de_to_nl,net_flow_de_nl,flow_nl_to_be,...,humidity,temperature_forecast,wind_speed_forecast,cloud_cover_forecast,precipitation_forecast,solar_radiation_forecast,pressure_forecast,wind_gusts_forecast,dew_point_forecast,humidity_forecast
0,2021-01-01 00:00:00+00:00,48.19,48.19,48.19,48.19,19.125,0.0,2193.3700,2193.3700,1418.4175,...,95.0,0.1,5.9,40.0,0.0,0.0,1005.5,11.9,-0.6,95.0
1,2021-01-01 01:00:00+00:00,44.68,44.68,44.68,44.68,19.125,0.0,2711.7125,2711.7125,1371.6950,...,95.0,0.6,6.2,83.0,0.0,0.0,1005.6,10.8,-0.2,95.0
2,2021-01-01 02:00:00+00:00,42.92,42.92,42.92,42.92,19.125,0.0,2661.7350,2661.7350,662.2700,...,94.0,0.7,5.5,54.0,0.0,0.0,1005.8,10.4,-0.1,94.0
3,2021-01-01 03:00:00+00:00,40.39,40.39,40.39,40.39,19.125,0.0,2469.2525,2469.2525,334.5325,...,94.0,-0.4,6.1,54.0,0.0,0.0,1006.0,11.9,-1.2,94.0
4,2021-01-01 04:00:00+00:00,40.20,40.20,40.20,40.20,19.125,0.0,2749.8375,2749.8375,626.7025,...,94.0,-0.3,7.7,77.0,0.0,0.0,1005.9,13.0,-1.2,94.0


In [148]:
import pandas as pd
from functools import reduce
from src.db.connection import get_engine

engine = get_engine()

START = "2021-01-01"
END = "2026-01-01"


def load_and_resample_hourly(table_name, value_cols, agg="mean"):
    df = pd.read_sql(
        f"""
        SELECT timestamp, {", ".join(value_cols)}
        FROM {table_name}
        WHERE timestamp >= '{START}'
          AND timestamp < '{END}'
        ORDER BY timestamp
        """,
        engine,
    )

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

    for col in value_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = (
        df
        .drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .set_index("timestamp")
    )

    df_hourly = (
        df
        .resample("1h")[value_cols]
        .agg(agg)
        .reset_index()
    )

    print(f"\n=== {table_name} ===")
    print("Shape:", df_hourly.shape)
    print("Time range:", df_hourly["timestamp"].min(), "→", df_hourly["timestamp"].max())
    print("Timestamp diff:")
    print(df_hourly["timestamp"].diff().value_counts().head())
    print("NaNs:")
    print(df_hourly[value_cols].isna().sum())

    return df_hourly


imbalance_h = load_and_resample_hourly(
    "raw_imbalance_price_15min",
    ["imbalance_price_long", "imbalance_price_short"],
)

load_h = load_and_resample_hourly(
    "raw_load_15min",
    ["load_mw"],
)

load_forecast_h = load_and_resample_hourly(
    "raw_load_forecast_15min",
    ["load_forecast"],
)

solar_h = load_and_resample_hourly(
    "raw_solar_generation_15min",
    ["solar_mw"],
)

solar_forecast_h = load_and_resample_hourly(
    "raw_solar_forecast_15min",
    ["solar_forecast_mw"],
)

wind_h = load_and_resample_hourly(
    "raw_wind_generation_15min",
    ["wind_mw"],
)

wind_forecast_h = load_and_resample_hourly(
    "raw_wind_forecast_15min",
    ["wind_forecast_mw"],
)


dfs_h = [
    imbalance_h,
    load_h,
    load_forecast_h,
    solar_h,
    solar_forecast_h,
    wind_h,
    wind_forecast_h,
]

master_generation_load_hourly = reduce(
    lambda left, right: pd.merge(left, right, on="timestamp", how="outer"),
    dfs_h,
)

master_generation_load_hourly = (
    master_generation_load_hourly
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("\n=== FINAL HOURLY DATASET ===")
print("Shape:", master_generation_load_hourly.shape)

print("\nTimestamp diff:")
print(master_generation_load_hourly["timestamp"].diff().value_counts())

print("\nDuplicates:")
print(master_generation_load_hourly["timestamp"].duplicated().sum())

print("\nNaN share:")
display(master_generation_load_hourly.isna().mean().sort_values(ascending=False))

display(master_generation_load_hourly.head())
display(master_generation_load_hourly.tail())


=== raw_imbalance_price_15min ===
Shape: (43823, 3)
Time range: 2021-01-01 00:00:00+00:00 → 2025-12-31 22:00:00+00:00
Timestamp diff:
timestamp
0 days 01:00:00    43822
Name: count, dtype: int64
NaNs:
imbalance_price_long     0
imbalance_price_short    0
dtype: int64

=== raw_load_15min ===
Shape: (43823, 2)
Time range: 2021-01-01 00:00:00+00:00 → 2025-12-31 22:00:00+00:00
Timestamp diff:
timestamp
0 days 01:00:00    43822
Name: count, dtype: int64
NaNs:
load_mw    0
dtype: int64

=== raw_load_forecast_15min ===
Shape: (43823, 2)
Time range: 2021-01-01 00:00:00+00:00 → 2025-12-31 22:00:00+00:00
Timestamp diff:
timestamp
0 days 01:00:00    43822
Name: count, dtype: int64
NaNs:
load_forecast    0
dtype: int64

=== raw_solar_generation_15min ===
Shape: (43823, 2)
Time range: 2021-01-01 00:00:00+00:00 → 2025-12-31 22:00:00+00:00
Timestamp diff:
timestamp
0 days 01:00:00    43822
Name: count, dtype: int64
NaNs:
solar_mw    0
dtype: int64

=== raw_solar_forecast_15min ===
Shape: (43823, 2)


solar_forecast_mw        0.000548
wind_forecast_mw         0.000548
timestamp                0.000000
imbalance_price_long     0.000000
imbalance_price_short    0.000000
load_mw                  0.000000
load_forecast            0.000000
solar_mw                 0.000000
wind_mw                  0.000000
dtype: float64

,timestamp,imbalance_price_long,imbalance_price_short,load_mw,load_forecast,solar_mw,solar_forecast_mw,wind_mw,wind_forecast_mw
0,2021-01-01 00:00:00+00:00,38.9300,38.9300,11313.1375,12044.455,0.5000,0.0,157.1075,273.00
1,2021-01-01 01:00:00+00:00,17.6025,17.6025,10925.1475,11738.760,0.5150,0.0,175.4475,305.25
2,2021-01-01 02:00:00+00:00,-14.1325,-14.1325,10576.8200,11606.410,0.5350,0.0,209.6750,340.25
3,2021-01-01 03:00:00+00:00,22.8150,22.8150,10371.1225,11677.355,0.5225,0.0,238.0125,342.75
4,2021-01-01 04:00:00+00:00,19.5925,19.5925,10420.2075,11972.285,0.5350,0.0,270.6150,336.25


,timestamp,imbalance_price_long,imbalance_price_short,load_mw,load_forecast,solar_mw,solar_forecast_mw,wind_mw,wind_forecast_mw
43818,2025-12-31 18:00:00+00:00,0.7850,0.7850,15378.79125,14675.88425,1.15975,0.0,1527.34900,2739.25
43819,2025-12-31 19:00:00+00:00,37.2775,37.2775,14421.90525,13869.51050,1.14675,0.0,1706.23450,2813.50
43820,2025-12-31 20:00:00+00:00,34.5375,63.5125,13749.90950,12952.07275,1.13150,0.0,1682.52075,2904.00
43821,2025-12-31 21:00:00+00:00,-14.8975,-14.8975,13318.43625,12017.94850,1.13825,0.0,1563.56625,3055.25
43822,2025-12-31 22:00:00+00:00,-20.0450,-20.0450,13008.82500,11128.98425,1.21175,0.0,1644.11175,3356.00


In [149]:
forecast_missing = master_generation_load_hourly.loc[
    master_generation_load_hourly[["solar_forecast_mw", "wind_forecast_mw"]].isna().any(axis=1),
    ["timestamp", "solar_forecast_mw", "wind_forecast_mw"]
]

display(forecast_missing)

forecast_cols = [
    "solar_forecast_mw",
    "wind_forecast_mw",
]

master_generation_load_hourly[forecast_cols] = (
    master_generation_load_hourly[forecast_cols]
    .ffill()
    .bfill()
)

print(master_generation_load_hourly[forecast_cols].isna().sum())

display(
    master_generation_load_hourly.isna()
    .mean()
    .sort_values(ascending=False)
)

,timestamp,solar_forecast_mw,wind_forecast_mw
25127,2023-11-13 23:00:00+00:00,NaN,NaN
25128,2023-11-14 00:00:00+00:00,NaN,NaN
25129,2023-11-14 01:00:00+00:00,NaN,NaN
25130,2023-11-14 02:00:00+00:00,NaN,NaN
25131,2023-11-14 03:00:00+00:00,NaN,NaN
25132,2023-11-14 04:00:00+00:00,NaN,NaN
25133,2023-11-14 05:00:00+00:00,NaN,NaN
25134,2023-11-14 06:00:00+00:00,NaN,NaN
25135,2023-11-14 07:00:00+00:00,NaN,NaN
25136,2023-11-14 08:00:00+00:00,NaN,NaN


solar_forecast_mw    0
wind_forecast_mw     0
dtype: int64


timestamp                0.0
imbalance_price_long     0.0
imbalance_price_short    0.0
load_mw                  0.0
load_forecast            0.0
solar_mw                 0.0
solar_forecast_mw        0.0
wind_mw                  0.0
wind_forecast_mw         0.0
dtype: float64

In [150]:
# какие колонки добавляем
gen_load_cols = [
    "imbalance_price_long",
    "imbalance_price_short",
    "load_mw",
    "load_forecast",
    "solar_mw",
    "solar_forecast_mw",
    "wind_mw",
    "wind_forecast_mw",
]

# убрать старые версии, если уже добавлялись
master = master.drop(
    columns=[c for c in gen_load_cols if c in master.columns],
    errors="ignore",
)

# timestamp types
master["timestamp"] = pd.to_datetime(master["timestamp"], utc=True)

hourly_tables = [
    ("imbalance_h", imbalance_h, ["imbalance_price_long", "imbalance_price_short"]),
    ("load_h", load_h, ["load_mw"]),
    ("load_forecast_h", load_forecast_h, ["load_forecast"]),
    ("solar_h", solar_h, ["solar_mw"]),
    ("solar_forecast_h", solar_forecast_h, ["solar_forecast_mw"]),
    ("wind_h", wind_h, ["wind_mw"]),
    ("wind_forecast_h", wind_forecast_h, ["wind_forecast_mw"]),
]

for name, df_add, cols in hourly_tables:
    print(f"\n=== Merging {name} ===")

    df_add = df_add.copy()
    df_add["timestamp"] = pd.to_datetime(df_add["timestamp"], utc=True)

    master = master.merge(
        df_add[["timestamp"] + cols],
        on="timestamp",
        how="left",
    )

    print("Master shape:", master.shape)
    print("Duplicates:", master["timestamp"].duplicated().sum())
    print("NaN share:")
    display(master[cols].isna().mean())

master = master.sort_values("timestamp").reset_index(drop=True)

print("\n=== FINAL CHECK ===")
print("Timestamp diff:")
print(master["timestamp"].diff().value_counts())

print("\nDuplicates:")
print(master["timestamp"].duplicated().sum())

print("\nNaN share added cols:")
display(master[gen_load_cols].isna().mean().sort_values(ascending=False))

display(master.head())
display(master.tail())


=== Merging imbalance_h ===
Master shape: (43820, 29)
Duplicates: 0
NaN share:


imbalance_price_long     0.000023
imbalance_price_short    0.000023
dtype: float64


=== Merging load_h ===
Master shape: (43820, 30)
Duplicates: 0
NaN share:


load_mw    0.000023
dtype: float64


=== Merging load_forecast_h ===
Master shape: (43820, 31)
Duplicates: 0
NaN share:


...

wind_forecast_mw    0.000571
dtype: float64


=== FINAL CHECK ===
Timestamp diff:
timestamp
0 days 01:00:00    43815
0 days 02:00:00        4
Name: count, dtype: int64

Duplicates:
0

NaN share added cols:


solar_forecast_mw        0.000571
wind_forecast_mw         0.000571
imbalance_price_long     0.000023
imbalance_price_short    0.000023
load_mw                  0.000023
load_forecast            0.000023
solar_mw                 0.000023
wind_mw                  0.000023
dtype: float64

,timestamp,nl_day_ahead_price,be_day_ahead_price,de_day_ahead_price,fr_day_ahead_price,gas_price,flow_nl_to_de,flow_de_to_nl,net_flow_de_nl,flow_nl_to_be,...,dew_point_forecast,humidity_forecast,imbalance_price_long,imbalance_price_short,load_mw,load_forecast,solar_mw,solar_forecast_mw,wind_mw,wind_forecast_mw
0,2021-01-01 00:00:00+00:00,48.19,48.19,48.19,48.19,19.125,0.0,2193.3700,2193.3700,1418.4175,...,-0.6,95.0,38.9300,38.9300,11313.1375,12044.455,0.5000,0.0,157.1075,273.00
1,2021-01-01 01:00:00+00:00,44.68,44.68,44.68,44.68,19.125,0.0,2711.7125,2711.7125,1371.6950,...,-0.2,95.0,17.6025,17.6025,10925.1475,11738.760,0.5150,0.0,175.4475,305.25
2,2021-01-01 02:00:00+00:00,42.92,42.92,42.92,42.92,19.125,0.0,2661.7350,2661.7350,662.2700,...,-0.1,94.0,-14.1325,-14.1325,10576.8200,11606.410,0.5350,0.0,209.6750,340.25
3,2021-01-01 03:00:00+00:00,40.39,40.39,40.39,40.39,19.125,0.0,2469.2525,2469.2525,334.5325,...,-1.2,94.0,22.8150,22.8150,10371.1225,11677.355,0.5225,0.0,238.0125,342.75
4,2021-01-01 04:00:00+00:00,40.20,40.20,40.20,40.20,19.125,0.0,2749.8375,2749.8375,626.7025,...,-1.2,94.0,19.5925,19.5925,10420.2075,11972.285,0.5350,0.0,270.6150,336.25


,timestamp,nl_day_ahead_price,be_day_ahead_price,de_day_ahead_price,fr_day_ahead_price,gas_price,flow_nl_to_de,flow_de_to_nl,net_flow_de_nl,flow_nl_to_be,...,dew_point_forecast,humidity_forecast,imbalance_price_long,imbalance_price_short,load_mw,load_forecast,solar_mw,solar_forecast_mw,wind_mw,wind_forecast_mw
43815,2025-12-31 19:00:00+00:00,82.9975,91.905,86.8025,92.0000,NaN,794.70,1689.27,894.57,3011.908,...,2.4,92.0,37.2775,37.2775,14421.90525,13869.51050,1.14675,0.0,1706.23450,2813.50
43816,2025-12-31 20:00:00+00:00,79.3525,83.710,79.7975,84.9975,NaN,398.74,1504.24,1105.50,3305.262,...,1.5,86.0,34.5375,63.5125,13749.90950,12952.07275,1.13150,0.0,1682.52075,2904.00
43817,2025-12-31 21:00:00+00:00,82.0475,84.650,81.3900,86.0975,NaN,35.29,1418.96,1383.67,2943.922,...,0.6,84.0,-14.8975,-14.8975,13318.43625,12017.94850,1.13825,0.0,1563.56625,3055.25
43818,2025-12-31 22:00:00+00:00,72.6075,80.495,76.4475,80.3450,NaN,0.00,1763.83,1763.83,3260.786,...,1.2,89.0,-20.0450,-20.0450,13008.82500,11128.98425,1.21175,0.0,1644.11175,3356.00
43819,2025-12-31 23:00:00+00:00,72.9100,106.300,63.6400,95.9500,NaN,NaN,NaN,NaN,NaN,...,1.2,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [155]:
# ── Save weather forecast and actual data separately ──
import os

output_path = "data/processed"
os.makedirs(output_path, exist_ok=True)

# 1. Weather forecast data (from raw_weather_forecast_hourly)
weather_forecast_to_save = weather_forecast.copy()

# ensure timestamp is timezone-naive for cleaner CSV
if weather_forecast_to_save["timestamp"].dt.tz is not None:
    weather_forecast_to_save["timestamp"] = weather_forecast_to_save["timestamp"].dt.tz_localize(None)

weather_forecast_path = os.path.join(output_path, "weather_forecast_hourly_2021_2025.csv")
weather_forecast_to_save.to_csv(weather_forecast_path, index=False)
print(f"Saved weather forecast to: {weather_forecast_path}")
print(f"  Shape: {weather_forecast_to_save.shape}")
print(f"  Columns: {list(weather_forecast_to_save.columns)}")
print(f"  Date range: {weather_forecast_to_save['timestamp'].min()} → {weather_forecast_to_save['timestamp'].max()}")


# 2. Weather actual data (from raw_weather_actual_hourly)
weather_actual_to_save = weather_actual.copy()

if weather_actual_to_save["timestamp"].dt.tz is not None:
    weather_actual_to_save["timestamp"] = weather_actual_to_save["timestamp"].dt.tz_localize(None)

weather_actual_path = os.path.join(output_path, "weather_actual_hourly_2021_2025.csv")
weather_actual_to_save.to_csv(weather_actual_path, index=False)
print(f"\nSaved weather actual to: {weather_actual_path}")
print(f"  Shape: {weather_actual_to_save.shape}")
print(f"  Columns: {list(weather_actual_to_save.columns)}")
print(f"  Date range: {weather_actual_to_save['timestamp'].min()} → {weather_actual_to_save['timestamp'].max()}")


# 3. Optional: also save Open Meteo D+1 forecasts if they exist in workspace
try:
    if 'df_fc' in dir() and df_fc is not None:
        df_fc_save = df_fc.copy()
        if df_fc_save.index.name == 'timestamp' or 'timestamp' in df_fc_save.columns:
            if 'timestamp' not in df_fc_save.columns:
                df_fc_save = df_fc_save.reset_index()
        if df_fc_save['timestamp'].dt.tz is not None:
            df_fc_save['timestamp'] = df_fc_save['timestamp'].dt.tz_localize(None)
        
        openmeteo_path = os.path.join(output_path, "weather_openmeteo_forecast_2023_2025.csv")
        df_fc_save.to_csv(openmeteo_path, index=False)
        print(f"\nSaved Open Meteo forecasts to: {openmeteo_path}")
        print(f"  Shape: {df_fc_save.shape}")
except NameError:
    print("\nNote: df_fc (Open Meteo) not found in workspace — skipping")

print("\n✅ All weather files saved successfully!")

Saved weather forecast to: data/processed/weather_forecast_hourly_2021_2025.csv
  Shape: (61368, 10)
  Columns: ['timestamp', 'temperature_forecast', 'wind_speed_forecast', 'cloud_cover_forecast', 'precipitation_forecast', 'solar_radiation_forecast', 'pressure_forecast', 'wind_gusts_forecast', 'dew_point_forecast', 'humidity_forecast']
  Date range: 2019-01-01 00:00:00 → 2025-12-31 23:00:00

Saved weather actual to: data/processed/weather_actual_hourly_2021_2025.csv
  Shape: (43824, 7)
  Columns: ['timestamp', 'temperature_c', 'wind_ms', 'solar_radiation', 'cloud_cover', 'precipitation', 'humidity']
  Date range: 2021-01-01 00:00:00 → 2025-12-31 23:00:00

✅ All weather files saved successfully!
